# AiStock V8 全链路测试 Notebook

**测试范围**: 数据获取 → 计算模块 → Plotly可视化  
**版本**: V8.0 | **Plotly**: 6.7.0  
**日期**: 2026-03-05

---

## 测试目录

| 部分 | 内容 | 状态 |
|------|------|------|
| Part 0 | 环境初始化 | ⬜ |
| Part 1 | 数据获取模块测试 | ⬜ |
| Part 2 | 计算模块测试 | ⬜ |
| Part 3 | Plotly可视化模块测试 | ⬜ |


## Part 0: 环境初始化

添加项目根目录到 `sys.path`，检查依赖，导入公共模块。


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 0.1: 添加项目根目录到 sys.path
# ═══════════════════════════════════════════════════════════════
import sys
import os
from pathlib import Path

# 项目根目录 = notebooks/ 的父目录
PROJECT_ROOT = str(Path().resolve().parent)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
    print(f"✅ 已添加项目根目录到 sys.path: {PROJECT_ROOT}")
else:
    print(f"✅ 项目根目录已在 sys.path 中: {PROJECT_ROOT}")

# notebooks 目录
NOTEBOOK_DIR = str(Path().resolve())
print(f"📁 Notebook 目录: {NOTEBOOK_DIR}")
print(f"📁 项目根目录: {PROJECT_ROOT}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 0.2: 检查依赖
# ═══════════════════════════════════════════════════════════════
import importlib

dependencies = {
    "pytdx": "pytdx (通达信接口)",
    "akshare": "akshare (AKShare数据接口)",
    "pandas": "pandas (数据处理)",
    "numpy": "numpy (数值计算)",
    "plotly": "plotly (可视化, 要求6.7.0)",
    "sqlalchemy": "sqlalchemy (数据库ORM)",
    "openpyxl": "openpyxl (Excel读取)",
    "nbformat": "nbformat (Notebook格式)",
}

print("=" * 60)
print("依赖检查")
print("=" * 60)

for module_name, desc in dependencies.items():
    try:
        mod = importlib.import_module(module_name)
        version = getattr(mod, "__version__", "未知版本")
        print(f"  ✅ {desc}: {version}")
    except ImportError:
        print(f"  ❌ {desc}: 未安装")

# 特殊检查 Plotly 版本
try:
    import plotly
    plotly_version = plotly.__version__
    major, minor = [int(x) for x in plotly_version.split(".")[:2]]
    if major >= 6 and minor >= 0:
        print(f"\n🎯 Plotly 版本 {plotly_version} 满足 >= 6.0 要求")
    else:
        print(f"\n⚠️ Plotly 版本 {plotly_version} 不满足 >= 6.0 要求, 建议升级")
except Exception as e:
    print(f"\n❌ Plotly 版本检查失败: {e}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 0.3: 导入公共模块
# ═══════════════════════════════════════════════════════════════
import time
import logging
import pandas as pd
import numpy as np
from datetime import datetime

# 配置日志
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s',
    datefmt='%H:%M:%S'
)

# 计时装饰器
def timed(label=""):
    # 计时上下文管理器
    class _Timer:
        def __init__(self, lbl):
            self.label = lbl
            self.start = 0
            self.elapsed_ms = 0
        def __enter__(self):
            self.start = time.time()
            return self
        def __exit__(self, *args):
            self.elapsed_ms = (time.time() - self.start) * 1000
            print(f"⏱️  {self.label}: {self.elapsed_ms:.0f}ms")
    return _Timer(label)

print("✅ 公共模块导入完成")
print(f"   pandas: {pd.__version__}")
print(f"   numpy: {np.__version__}")


---

## Part 1: 数据获取模块测试

### 1.1 xlsx代码表加载与code对齐验证

**关键问题**: xlsx文件有两个代码列：
- `code`: TDX内部代码（如 `10009633`、`IO8Q0668`、`90005865`）
- `code_name`: 人类可读代码（如 `510050C3A02750`、`IO2602-P-4000`）

pytdx的 `get_instrument_bars(category, market, code, start, count)` 的 `code` 参数
**必须使用xlsx的 `code` 列值**，而非 `code_name` 列值。


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 1.1: xlsx代码表加载与code对齐验证
# ═══════════════════════════════════════════════════════════════
xlsx_path = os.path.join(NOTEBOOK_DIR, "tdx基金期货期权代码表.xlsx")
print(f"📂 xlsx路径: {xlsx_path}")

try:
    code_df = pd.read_excel(xlsx_path, engine="openpyxl")
    print(f"✅ xlsx加载成功! 共 {len(code_df)} 行")
    print(f"\n📋 列名: {list(code_df.columns)}")
    print(f"\n📊 前5行预览:")
    display(code_df.head())
except Exception as e:
    print(f"❌ xlsx加载失败: {e}")
    code_df = pd.DataFrame()


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 1.1b: market_code 分布统计
# ═══════════════════════════════════════════════════════════════
if not code_df.empty:
    print("=" * 60)
    print("market_code 分布统计")
    print("=" * 60)
    
    mc_dist = code_df['market_code'].value_counts().sort_index()
    for mc, count in mc_dist.items():
        # 根据market_code映射说明
        mc_names = {
            7: "中金所期权(IO/HO/MO)",
            8: "上交所ETF期权",
            9: "深交所ETF期权",
            28: "郑商所",
            30: "上期所",
            33: "基金",
            47: "中金所期货",
        }
        name = mc_names.get(mc, "其他")
        print(f"  market_code={mc:>3d} ({name}): {count:>6d} 条")
    
    print(f"\n📊 总计: {len(code_df)} 条")
    
    # category分布
    print("\n" + "=" * 60)
    print("category 分布统计")
    print("=" * 60)
    cat_dist = code_df['category'].value_counts().sort_index()
    cat_names = {3: "期货", 8: "基金", 10: "其他", 12: "期权"}
    for cat, count in cat_dist.items():
        name = cat_names.get(cat, "其他")
        print(f"  category={cat:>3d} ({name}): {count:>6d} 条")
else:
    print("⚠️ code_df 为空, 跳过统计")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 1.1c: 各关键市场的样本行展示
# ═══════════════════════════════════════════════════════════════
if not code_df.empty:
    key_markets = {
        8: "上交所ETF期权 (market_code=8)",
        9: "深交所ETF期权 (market_code=9)",
        7: "中金所期权 (market_code=7)",
        47: "中金所期货 (market_code=47)",
        33: "基金 (market_code=33)",
        28: "郑商所 (market_code=28)",
        30: "上期所 (market_code=30)",
    }
    
    for mc, desc in key_markets.items():
        subset = code_df[code_df['market_code'] == mc]
        if not subset.empty:
            sample = subset.head(3)
            print(f"\n📋 {desc} — 共 {len(subset)} 条, 展示前3条:")
            for _, row in sample.iterrows():
                print(f"   code={row['code']:<15s} | code_name={row['code_name']:<25s} | market_name={row.get('market_name', 'N/A')}")
        else:
            print(f"\n📋 {desc} — 无数据")
else:
    print("⚠️ code_df 为空, 跳过样本展示")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 1.1d: ★★★ 关键对齐测试 — code vs code_name ★★★
# ═══════════════════════════════════════════════════════════════
# 验证: xlsx的'code'列和'code_name'列的关系
# pytdx get_instrument_bars 的 code 参数必须使用 xlsx 的 'code' 列值

if not code_df.empty:
    print("=" * 70)
    print("★★★ 关键对齐测试: code vs code_name ★★★")
    print("=" * 70)
    
    # 展示各市场的 code 和 code_name 对比
    for mc in [8, 9, 7, 47]:
        subset = code_df[code_df['market_code'] == mc]
        if subset.empty:
            continue
        
        mc_names = {8: "上交所ETF期权", 9: "深交所ETF期权", 7: "中金所期权", 47: "中金所期货"}
        print(f"\n📌 {mc_names.get(mc, f'market_code={mc}')}:")
        print(f"   code列示例:    {list(subset['code'].head(5))}")
        print(f"   code_name列示例: {list(subset['code_name'].head(5))}")
        
        # 检查是否所有code都以数字开头(内部代码) vs code_name包含字母
        code_starts_digit = sum(1 for c in subset['code'] if str(c)[0].isdigit())
        codename_has_letter = sum(1 for c in subset['code_name'] if any(ch.isalpha() for ch in str(c)))
        
        print(f"   code列以数字开头: {code_starts_digit}/{len(subset)} ({code_starts_digit/len(subset)*100:.1f}%)")
        print(f"   code_name列含字母: {codename_has_letter}/{len(subset)} ({codename_has_letter/len(subset)*100:.1f}%)")
    
    # 关键结论
    print("\n" + "=" * 70)
    print("🔑 关键结论:")
    print("   - xlsx 'code' 列 = TDX内部代码 (纯数字或字母数字混合)")
    print("   - xlsx 'code_name' 列 = 人类可读代码 (含期权格式如510050C6A02850)")
    print("   - pytdx get_instrument_bars() 必须使用 'code' 列值!")
    print("=" * 70)
else:
    print("⚠️ code_df 为空, 跳过对齐测试")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 1.1e: 构建查找字典 {market_code: {code: code_name}}
# ═══════════════════════════════════════════════════════════════
code_lookup = {}  # {market_code: {code: code_name}}
code_reverse_lookup = {}  # {market_code: {code_name: code}}

if not code_df.empty:
    for mc in code_df['market_code'].unique():
        subset = code_df[code_df['market_code'] == mc]
        code_lookup[mc] = dict(zip(subset['code'].astype(str), subset['code_name'].astype(str)))
        code_reverse_lookup[mc] = dict(zip(subset['code_name'].astype(str), subset['code'].astype(str)))
    
    print("✅ 查找字典构建完成!")
    for mc, mapping in code_lookup.items():
        print(f"   market_code={mc}: {len(mapping)} 条映射")
    
    # 保存到全局变量供后续使用
    # 示例: 根据code_name查找code
    example_mc = 8
    if example_mc in code_reverse_lookup:
        example_cname = list(code_reverse_lookup[example_mc].keys())[0]
        example_code = code_reverse_lookup[example_mc][example_cname]
        print(f"\n📌 示例: market_code=8, code_name='{example_cname}' → code='{example_code}'")
else:
    print("⚠️ code_df 为空, 查找字典未构建")


### 1.2 TDX标准端口测试 (7709)

测试通达信标准端口的连接和数据获取能力。
- 标准端口: `180.153.18.170:7709`
- 用途: 股票/指数K线、实时行情


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 1.2a: 直连pytdx标准端口测试
# ═══════════════════════════════════════════════════════════════
try:
    from pytdx.hq import TdxHq_API
    
    std_api = TdxHq_API()
    with timed("标准端口连接 (7709)"):
        conn_result = std_api.connect("180.153.18.170", 7709)
    
    if conn_result:
        print("✅ 标准端口连接成功!")
        
        # 获取证券数量
        count = std_api.get_security_count(0)  # 深圳市场
        print(f"   深圳市场证券数量: {count}")
        count_sh = std_api.get_security_count(1)  # 上海市场
        print(f"   上海市场证券数量: {count_sh}")
        
        std_api.disconnect()
    else:
        print("❌ 标准端口连接失败!")
except ImportError:
    print("❌ pytdx 未安装, 跳过标准端口测试")
except Exception as e:
    print(f"❌ 标准端口测试异常: {e}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 1.2b: 指数K线测试
# ═══════════════════════════════════════════════════════════════
index_tests = [
    {"code": "000001", "market": 1, "name": "上证指数"},
    {"code": "000300", "market": 1, "name": "沪深300"},
    {"code": "399006", "market": 0, "name": "创业板指"},
    {"code": "000905", "market": 1, "name": "中证500"},
    {"code": "000852", "market": 1, "name": "中证1000"},
]

try:
    from pytdx.hq import TdxHq_API
    
    api = TdxHq_API()
    api.connect("180.153.18.170", 7709)
    
    for idx_cfg in index_tests:
        try:
            with timed(f"获取 {idx_cfg['name']} K线"):
                df = api.get_index_bars(7, idx_cfg['market'], idx_cfg['code'], 0, 10)
            
            if df is not None and not df.empty:
                latest = df.iloc[-1]
                print(f"  ✅ {idx_cfg['name']}: 最新日期={latest.get('day', 'N/A')}, 收盘={latest.get('close', 'N/A')}")
            else:
                print(f"  ⚠️ {idx_cfg['name']}: 返回空数据")
        except Exception as e:
            print(f"  ❌ {idx_cfg['name']}: {e}")
    
    api.disconnect()
except Exception as e:
    print(f"❌ 指数K线测试失败: {e}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 1.2c: 实时行情测试
# ═══════════════════════════════════════════════════════════════
try:
    from pytdx.hq import TdxHq_API
    
    api = TdxHq_API()
    api.connect("180.153.18.170", 7709)
    
    # 批量获取实时行情
    quotes_params = [
        (1, "600000"),  # 浦发银行
        (1, "601318"),  # 中国平安
        (0, "000001"),  # 平安银行
    ]
    
    with timed("实时行情获取"):
        quotes = api.get_security_quotes(quotes_params)
    
    if quotes:
        print(f"✅ 获取到 {len(quotes)} 条实时行情:")
        for q in quotes:
            print(f"   {q.get('code', 'N/A')} {q.get('name', 'N/A')}: "
                  f"价格={q.get('price', 'N/A')} 涨跌={q.get('price', 0)-q.get('last_close', 0):.2f}")
    else:
        print("⚠️ 实时行情返回空")
    
    api.disconnect()
except Exception as e:
    print(f"❌ 实时行情测试失败: {e}")


### 1.3 TDX扩展端口测试 (7721)

测试通达信扩展端口的连接和数据获取能力。
- 扩展端口: `180.153.18.176:7721`
- 用途: 期货/期权K线、合约列表、宏观指标

**关键测试**: 期权合约发现与code对齐验证


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 1.3a: 扩展端口连接测试
# ═══════════════════════════════════════════════════════════════
try:
    from pytdx.exhq import TdxExHq_API
    
    ext_api = TdxExHq_API()
    with timed("扩展端口连接 (7721)"):
        conn_result = ext_api.connect("180.153.18.176", 7721)
    
    if conn_result:
        print("✅ 扩展端口连接成功!")
        
        # 获取合约总数
        total = ext_api.get_instrument_count()
        print(f"   合约总数: {total}")
        
        ext_api.disconnect()
    else:
        print("❌ 扩展端口连接失败!")
except ImportError:
    print("❌ pytdx 未安装, 跳过扩展端口测试")
except Exception as e:
    print(f"❌ 扩展端口连接异常: {e}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 1.3b: 期货K线测试
# ═══════════════════════════════════════════════════════════════
future_tests = [
    {"code": "IFM0", "market": 47, "name": "沪深300主力"},
    {"code": "IHM0", "market": 47, "name": "上证50主力"},
    {"code": "CUM0", "market": 30, "name": "沪铜主力"},
    {"code": "AUM0", "market": 30, "name": "沪金主力"},
]

try:
    from pytdx.exhq import TdxExHq_API
    
    api = TdxExHq_API()
    api.connect("180.153.18.176", 7721)
    
    for f_cfg in future_tests:
        try:
            with timed(f"获取 {f_cfg['name']} K线"):
                df = api.get_instrument_bars(7, f_cfg['market'], f_cfg['code'], 0, 5)
            
            if df is not None and not df.empty:
                latest = df.iloc[-1]
                print(f"  ✅ {f_cfg['name']}: 收盘={latest.get('close', 'N/A')}, 成交量={latest.get('vol', 'N/A')}")
            else:
                print(f"  ⚠️ {f_cfg['name']}: 返回空数据")
        except Exception as e:
            print(f"  ❌ {f_cfg['name']}: {e}")
    
    api.disconnect()
except Exception as e:
    print(f"❌ 期货K线测试失败: {e}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 1.3c: ★★★ 合约列表发现测试 ★★★
# 测试 get_instrument_info 对 markets 8, 9, 48 的返回
# 对比返回的 'code' 列与 xlsx 的 'code' 列
# ═══════════════════════════════════════════════════════════════
try:
    from pytdx.exhq import TdxExHq_API
    
    api = TdxExHq_API()
    api.connect("180.153.18.176", 7721)
    
    # 获取全量合约列表
    total = api.get_instrument_count()
    print(f"📊 合约总数: {total}")
    
    # 分批获取
    all_instruments = []
    batch_size = 2000
    for start in range(0, total + batch_size, batch_size):
        df = api.get_instrument_info(start=start, count=batch_size)
        if df is not None and not df.empty:
            all_instruments.append(df)
    
    if all_instruments:
        inst_df = pd.concat(all_instruments, ignore_index=True)
        print(f"✅ 全量合约列表获取成功: {len(inst_df)} 条")
        print(f"   列名: {list(inst_df.columns)}")
        
        # 按market分组统计
        if 'market' in inst_df.columns:
            market_dist = inst_df['market'].value_counts().sort_index()
            print(f"\n📊 market 分布 (TDX扩展端口):")
            for m, count in market_dist.head(20).items():
                print(f"   market={m:>3d}: {count:>6d} 条")
    else:
        inst_df = pd.DataFrame()
        print("⚠️ 合约列表为空")
    
    api.disconnect()
except Exception as e:
    print(f"❌ 合约列表获取失败: {e}")
    inst_df = pd.DataFrame()


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 1.3d: ★★★ TDX API返回的code vs xlsx的code 对比 ★★★
# ═══════════════════════════════════════════════════════════════
# 关键验证: TDX扩展端口API返回的合约列表中, 'code'列与xlsx的'code'列是否一致?

if not inst_df.empty and not code_df.empty:
    print("=" * 70)
    print("★★★ TDX API code vs xlsx code 对齐验证 ★★★")
    print("=" * 70)
    
    # TDX扩展端口市场编号与xlsx market_code映射
    # 注意: TDX扩展端口中 market 48 = 中金期权, 但 xlsx 中 market_code=7
    # TDX扩展端口中 market 8 = 上交所ETF期权, 与 xlsx 一致
    # TDX扩展端口中 market 9 = 深交所ETF期权, 与 xlsx 一致
    
    market_mapping = {
        8: {"tdx_market": 8, "xlsx_market_code": 8, "desc": "上交所ETF期权"},
        9: {"tdx_market": 9, "xlsx_market_code": 9, "desc": "深交所ETF期权"},
        48: {"tdx_market": 48, "xlsx_market_code": 7, "desc": "中金所期权"},
    }
    
    for tdx_m, info in market_mapping.items():
        print(f"\n📌 {info['desc']} (TDX market={tdx_m}, xlsx market_code={info['xlsx_market_code']})")
        
        # TDX API返回的合约
        tdx_subset = inst_df[inst_df['market'] == tdx_m] if 'market' in inst_df.columns else pd.DataFrame()
        # xlsx中的合约
        xlsx_subset = code_df[code_df['market_code'] == info['xlsx_market_code']]
        
        if tdx_subset.empty:
            print(f"   ⚠️ TDX API无market={tdx_m}的数据")
            continue
        if xlsx_subset.empty:
            print(f"   ⚠️ xlsx无market_code={info['xlsx_market_code']}的数据")
            continue
        
        # 对比code列
        tdx_codes = set(str(c) for c in tdx_subset['code'].astype(str))
        xlsx_codes = set(str(c) for c in xlsx_subset['code'].astype(str))
        
        overlap = tdx_codes & xlsx_codes
        only_tdx = tdx_codes - xlsx_codes
        only_xlsx = xlsx_codes - tdx_codes
        
        print(f"   TDX API合约数: {len(tdx_codes)}")
        print(f"   xlsx合约数: {len(xlsx_codes)}")
        print(f"   重叠合约数: {len(overlap)} ({len(overlap)/max(len(tdx_codes),1)*100:.1f}%)")
        print(f"   仅TDX有: {len(only_tdx)}")
        print(f"   仅xlsx有: {len(only_xlsx)}")
        
        # 展示TDX API返回的code格式示例
        tdx_sample = list(tdx_subset['code'].astype(str).head(5))
        print(f"   TDX API code示例: {tdx_sample}")
        
        # 展示xlsx code格式示例
        xlsx_sample = list(xlsx_subset['code'].astype(str).head(5))
        print(f"   xlsx code示例: {xlsx_sample}")
        
        if overlap:
            print(f"   ✅ 有重叠! 证明xlsx 'code'列与TDX API的'code'列格式一致")
        else:
            print(f"   ⚠️ 无重叠, 需进一步分析格式差异")
else:
    print("⚠️ 数据不足, 跳过对齐验证")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 1.3e: ★★★ 期权K线测试 — 使用xlsx code列 ★★★
# 关键测试: 使用xlsx的'code'列值调用pytdx get_instrument_bars
# 同时尝试使用'code_name'列值, 对比哪种格式能获取数据
# ═══════════════════════════════════════════════════════════════
try:
    from pytdx.exhq import TdxExHq_API
    
    api = TdxExHq_API()
    api.connect("180.153.18.176", 7721)
    
    print("=" * 70)
    print("★★★ 期权K线 code vs code_name 对比测试 ★★★")
    print("=" * 70)
    
    # 测试各市场的期权合约
    test_cases = []
    
    # 上交所ETF期权 (market=8)
    sh_opt = code_df[code_df['market_code'] == 8]
    if not sh_opt.empty:
        sample = sh_opt.iloc[0]
        test_cases.append({
            "desc": "上交所ETF期权",
            "tdx_market": 8,
            "code": str(sample['code']),
            "code_name": str(sample['code_name']),
        })
    
    # 深交所ETF期权 (market=9)
    sz_opt = code_df[code_df['market_code'] == 9]
    if not sz_opt.empty:
        sample = sz_opt.iloc[0]
        test_cases.append({
            "desc": "深交所ETF期权",
            "tdx_market": 9,
            "code": str(sample['code']),
            "code_name": str(sample['code_name']),
        })
    
    # 中金所期权 (market_code=7, 但TDX扩展端口market=48)
    cffex_opt = code_df[code_df['market_code'] == 7]
    if not cffex_opt.empty:
        sample = cffex_opt.iloc[0]
        test_cases.append({
            "desc": "中金所期权",
            "tdx_market": 48,  # 注意: 中金期权在TDX扩展端口用market=48
            "code": str(sample['code']),
            "code_name": str(sample['code_name']),
        })
    
    for tc in test_cases:
        print(f"\n📌 {tc['desc']}:")
        print(f"   xlsx code='{tc['code']}' | code_name='{tc['code_name']}' | TDX market={tc['tdx_market']}")
        
        # 尝试1: 使用xlsx 'code'列值
        try:
            df_code = api.get_instrument_bars(7, tc['tdx_market'], tc['code'], 0, 3)
            if df_code is not None and not df_code.empty:
                print(f"   ✅ 使用 'code' 列值获取成功! 返回 {len(df_code)} 条K线")
            else:
                print(f"   ❌ 使用 'code' 列值获取失败: 返回空")
        except Exception as e:
            print(f"   ❌ 使用 'code' 列值获取异常: {e}")
        
        # 尝试2: 使用xlsx 'code_name'列值
        try:
            df_codename = api.get_instrument_bars(7, tc['tdx_market'], tc['code_name'], 0, 3)
            if df_codename is not None and not df_codename.empty:
                print(f"   ✅ 使用 'code_name' 列值获取成功! 返回 {len(df_codename)} 条K线")
            else:
                print(f"   ❌ 使用 'code_name' 列值获取失败: 返回空")
        except Exception as e:
            print(f"   ❌ 使用 'code_name' 列值获取异常: {e}")
    
    print("\n" + "=" * 70)
    print("🔑 结论: 哪种code格式能获取到K线数据, 就说明pytdx接受该格式")
    print("=" * 70)
    
    api.disconnect()
except Exception as e:
    print(f"❌ 期权K线测试失败: {e}")


### 1.4 AKShare外盘期货测试

测试 AKShare 海外期货数据获取能力。


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 1.4a: 直接测试 ak.futures_foreign_hist
# ═══════════════════════════════════════════════════════════════
try:
    import akshare as ak
    print(f"✅ akshare 版本: {ak.__version__}")
    
    # 测试3个核心品种
    test_symbols = [
        {"ak_code": "WTI原油", "name": "WTI原油"},
        {"ak_code": "COMEX黄金", "name": "COMEX黄金"},
        {"ak_code": "COMEX铜", "name": "COMEX铜"},
    ]
    
    for sym in test_symbols:
        try:
            with timed(f"获取 {sym['name']} 历史数据"):
                df = ak.futures_foreign_hist(symbol=sym['ak_code'])
            if df is not None and not df.empty:
                print(f"  ✅ {sym['name']}: {len(df)} 条数据, 最新={df.iloc[-1].get('日期', df.iloc[-1].get('date', 'N/A'))}")
            else:
                print(f"  ⚠️ {sym['name']}: 返回空数据")
        except Exception as e:
            print(f"  ❌ {sym['name']}: {e}")
            
except ImportError:
    print("❌ akshare 未安装")
except Exception as e:
    print(f"❌ AKShare测试失败: {e}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 1.4b: V8 AKAdapter模块测试
# ═══════════════════════════════════════════════════════════════
try:
    from data_service.ak_adapter import AKAdapter, FUTURES_SYMBOL_MAP, CORE_SYMBOLS
    
    ak_adapter = AKAdapter(rate_limit_interval=0.5, cache_ttl=60)
    print(f"✅ AKAdapter 初始化成功")
    print(f"   品种总数: {len(FUTURES_SYMBOL_MAP)}")
    print(f"   核心品种: {CORE_SYMBOLS}")
    
    # 健康检查
    health = ak_adapter.health_check()
    print(f"   健康检查: {health}")
    
    # 测试3个核心品种
    print(f"\n📊 核心品种批量测试 (3个):")
    for sym in CORE_SYMBOLS[:3]:
        try:
            with timed(f"  {sym}"):
                data = ak_adapter.get_futures_realtime(sym)
            if data:
                print(f"    ✅ {sym}: 价格={data.get('price', 'N/A')}, 涨跌幅={data.get('pct_change', 'N/A')}%")
            else:
                print(f"    ⚠️ {sym}: 返回None")
        except Exception as e:
            print(f"    ❌ {sym}: {e}")
    
except ImportError as e:
    print(f"❌ AKAdapter导入失败: {e}")
except Exception as e:
    print(f"❌ AKAdapter测试失败: {e}")


### 1.5 DatabaseReader测试

测试 PostgreSQL 数据库连接和 PE/PB 数据查询。


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 1.5: DatabaseReader测试
# ═══════════════════════════════════════════════════════════════
try:
    from data_service.database_reader import DatabaseReader
    
    # 使用fallback=True, 连接失败不会抛异常
    db_reader = DatabaseReader(fallback_on_error=True)
    
    # 健康检查
    health = db_reader.health_check()
    print(f"📊 DatabaseReader 健康检查: {health}")
    
    if db_reader.is_connected:
        print("✅ 数据库已连接!")
        
        # 尝试获取沪深300 PE数据
        try:
            pe_df = db_reader.get_index_pe("000300", days=10)
            if not pe_df.empty:
                print(f"   沪深300 PE数据: {len(pe_df)} 条")
                print(f"   最新PE: {pe_df.iloc[0].to_dict() if not pe_df.empty else 'N/A'}")
            else:
                print("   ⚠️ PE数据为空")
        except Exception as e:
            print(f"   ❌ PE数据查询失败: {e}")
    else:
        print("⚠️ 数据库未连接 (这是正常的, 如果没有本地PostgreSQL)")
        print("   系统将以fallback模式运行, PE/PB相关功能将使用默认值")
        
except ImportError as e:
    print(f"❌ DatabaseReader导入失败: {e}")
except Exception as e:
    print(f"❌ DatabaseReader测试失败: {e}")


### 1.6 TDXAdapter模块化测试

测试 V8 TDXAdapter 类的完整功能。


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 1.6: TDXAdapter模块化测试
# ═══════════════════════════════════════════════════════════════
try:
    from data_service.tdx_adapter import TDXAdapter, MarketType, BarCategory, MARKET_MAP
    
    tdx = TDXAdapter(pool_size=2, retry_count=2)
    print("✅ TDXAdapter 初始化成功")
    print(f"   市场映射表: {len(MARKET_MAP)} 个市场")
    
    # 1. 健康检查
    print("\n📊 健康检查:")
    with timed("双端口健康检查"):
        health = tdx.health_check()
    print(f"   标准端口(7709): {'✅' if health.get('standard_port') else '❌'}")
    print(f"   扩展端口(7721): {'✅' if health.get('extension_port') else '❌'}")
    
    # 2. 指数日线测试
    print("\n📊 指数日线测试:")
    for code, mt, name in [("000001", MarketType.INDEX_SH, "上证指数"),
                            ("000300", MarketType.INDEX_SH, "沪深300"),
                            ("399006", MarketType.INDEX_SZ, "创业板指")]:
        try:
            with timed(f"  {name}"):
                df = tdx.get_index_daily(code, market_type=mt, count=5)
            if not df.empty:
                print(f"    ✅ {name}: {len(df)} 条, 最新收盘={df.iloc[-1].get('close', 'N/A')}")
            else:
                print(f"    ⚠️ {name}: 返回空")
        except Exception as e:
            print(f"    ❌ {name}: {e}")
    
    # 3. 期货日线测试
    print("\n📊 期货日线测试:")
    for code, mt, name in [("IFM0", MarketType.FUTURE_ZJ, "沪深300主力"),
                            ("AUM0", MarketType.FUTURE_SH, "沪金主力")]:
        try:
            with timed(f"  {name}"):
                df = tdx.get_future_daily(code, market_type=mt, count=5)
            if not df.empty:
                print(f"    ✅ {name}: {len(df)} 条, 最新收盘={df.iloc[-1].get('close', 'N/A')}")
            else:
                print(f"    ⚠️ {name}: 返回空")
        except Exception as e:
            print(f"    ❌ {name}: {e}")
    
    # 4. 合约列表测试
    print("\n📊 合约列表测试 (扩展端口):")
    try:
        with timed("  中金期货合约列表"):
            inst_df = tdx.get_instrument_info(market=47)
        print(f"    ✅ 中金期货: {len(inst_df)} 条合约")
        if not inst_df.empty:
            print(f"    列名: {list(inst_df.columns)}")
    except Exception as e:
        print(f"    ❌ 合约列表: {e}")
    
    # 关闭连接
    tdx.close()
    print("\n✅ TDXAdapter 测试完成, 连接已关闭")
    
except ImportError as e:
    print(f"❌ TDXAdapter导入失败: {e}")
except Exception as e:
    print(f"❌ TDXAdapter测试失败: {e}")


---

## Part 2: 计算模块测试

### 2.1 OptionCodeParser测试

测试三种期权代码格式的解析能力：
1. 上交所ETF期权 (Market=8): `510050C6A02850`
2. 深交所ETF期权 (Market=9): `159901C6M003100A`
3. 中金所/商品期权: `IO2606-C-4000`


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 2.1a: OptionCodeParser基础测试 — 3种格式解析
# ═══════════════════════════════════════════════════════════════
try:
    from market_state_system.core.option_code_parser import OptionCodeParser, OptionContractInfo
    
    parser = OptionCodeParser()
    print(f"✅ OptionCodeParser 初始化成功: {parser}")
    
    # 测试用例
    test_cases = [
        {"code_name": "510050C6A02850", "market_code": 8, "desc": "上交所ETF期权"},
        {"code_name": "510300P6B03200", "market_code": 8, "desc": "上交所ETF期权(Put)"},
        {"code_name": "159901C6M003100A", "market_code": 9, "desc": "深交所ETF期权(含A后缀)"},
        {"code_name": "159915P6N002500", "market_code": 9, "desc": "深交所ETF期权(Put)"},
        {"code_name": "IO2606-C-4000", "market_code": 7, "desc": "中金所期权(Call)"},
        {"code_name": "HO2606-P-2400", "market_code": 7, "desc": "中金所期权(Put)"},
        {"code_name": "CU2606-C-100000", "market_code": 6, "desc": "商品期权"},
    ]
    
    print("\n📊 逐合约解析测试:")
    for tc in test_cases:
        info = parser.parse(tc["code_name"], tc["market_code"])
        if info:
            print(f"  ✅ {tc['desc']}: {info}")
        else:
            print(f"  ❌ {tc['desc']}: 解析失败")
    
    print(f"\n📊 解析统计: {parser.stats}")
    
except ImportError as e:
    print(f"❌ OptionCodeParser导入失败: {e}")
except Exception as e:
    print(f"❌ OptionCodeParser测试失败: {e}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 2.1b: 批量解析测试
# ═══════════════════════════════════════════════════════════════
try:
    from market_state_system.core.option_code_parser import OptionCodeParser
    
    parser = OptionCodeParser()
    
    # 构建批量测试数据
    batch_contracts = [
        {"code_name": "510050C6A02750", "market_code": 8},
        {"code_name": "510050P6M02800", "market_code": 8},
        {"code_name": "510300C6B03100", "market_code": 8},
        {"code_name": "159901C6M003100A", "market_code": 9},
        {"code_name": "159915P6N002500", "market_code": 9},
        {"code_name": "IO2606-C-4000", "market_code": 7},
        {"code_name": "IO2606-P-4000", "market_code": 7},
        {"code_name": "HO2606-C-2400", "market_code": 7},
        {"code_name": "CU2606-C-100000", "market_code": 6},
        {"code_name": "AG2606-P-10000", "market_code": 5},
    ]
    
    print(f"📊 批量解析测试: {len(batch_contracts)} 个合约")
    results = parser.parse_batch(batch_contracts)
    print(f"   成功解析: {len(results)} 个")
    
    for r in results:
        print(f"   {r.code_name}: {r.underlying} {r.direction.upper()} "
              f"{r.delivery_year}/{r.delivery_month:02d} K={r.strike_price} [{r.parse_source}]")
    
    print(f"\n📊 解析统计: {parser.stats}")
    
except Exception as e:
    print(f"❌ 批量解析测试失败: {e}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 2.1c: 代码构建测试 (反向操作: 解析→构建→验证)
# ═══════════════════════════════════════════════════════════════
try:
    from market_state_system.core.option_code_parser import OptionCodeParser
    
    parser = OptionCodeParser()
    
    # ETF期权构建测试
    print("📊 ETF期权代码构建测试:")
    
    # 构建510050看涨期权
    built_code = OptionCodeParser.build_etf_code(
        underlying="510050", direction="call",
        year=2026, month=1, strike_price=2.750,
    )
    print(f"   build_etf_code(510050, call, 2026, 1, 2.750) = '{built_code}'")
    
    # 反向解析验证
    info = parser.parse(built_code, market_code=8)
    if info:
        print(f"   反向解析: ✅ {info.underlying} {info.direction} "
              f"{info.delivery_year}/{info.delivery_month:02d} K={info.strike_price}")
    else:
        print(f"   反向解析: ❌ 失败")
    
    # CFFEX期权构建测试
    print("\n📊 CFFEX期权代码构建测试:")
    built_cffex = OptionCodeParser.build_cffex_code(
        variety="IO", year=2026, month=6, direction="call", strike_price=4000,
    )
    print(f"   build_cffex_code(IO, 2026, 6, call, 4000) = '{built_cffex}'")
    
    info2 = parser.parse(built_cffex, market_code=7)
    if info2:
        print(f"   反向解析: ✅ {info2.variety} {info2.direction} "
              f"{info2.delivery_year}/{info2.delivery_month:02d} K={info2.strike_price}")
    else:
        print(f"   反向解析: ❌ 失败")
    
    # 月份字母映射测试
    print("\n📊 月份字母映射测试:")
    for month in range(1, 13):
        call_letter = OptionCodeParser.get_month_letter(month, "call")
        put_letter = OptionCodeParser.get_month_letter(month, "put")
        print(f"   {month:>2d}月: Call={call_letter}, Put={put_letter}")
    
except Exception as e:
    print(f"❌ 代码构建测试失败: {e}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 2.1d: 使用xlsx code_name值进行解析测试
# ═══════════════════════════════════════════════════════════════
try:
    from market_state_system.core.option_code_parser import OptionCodeParser
    
    parser = OptionCodeParser()
    
    if not code_df.empty:
        print("📊 使用xlsx code_name值进行批量解析测试:")
        
        # 筛选期权数据
        option_df = code_df[code_df['category'] == 12]  # category=12为期权
        if option_df.empty:
            option_df = code_df[code_df['market_code'].isin([7, 8, 9])]
        
        if not option_df.empty:
            # 取每个市场最多10条
            sample_list = []
            for mc in [8, 9, 7]:
                mc_data = option_df[option_df['market_code'] == mc].head(10)
                for _, row in mc_data.iterrows():
                    sample_list.append({
                        "code_name": str(row['code_name']),
                        "market_code": row['market_code'],
                    })
            
            print(f"   测试样本数: {len(sample_list)}")
            results = parser.parse_batch(sample_list)
            print(f"   成功解析: {len(results)}/{len(sample_list)}")
            
            for r in results[:5]:
                print(f"   {r.code_name}: {r.underlying} {r.direction.upper()} "
                      f"K={r.strike_price} [{r.parse_source}]")
            
            print(f"\n   解析统计: {parser.stats}")
        else:
            print("   ⚠️ 期权数据为空")
    else:
        print("⚠️ code_df 为空, 跳过")
        
except Exception as e:
    print(f"❌ xlsx code_name解析测试失败: {e}")


### 2.2 期权PCR计算测试

测试 OptionPCREngine 的 PCR 计算流程。

**注意**: OptionPCREngine 依赖 TDXAdapter、ConfigService、CacheService，
在测试环境中可能因服务不可用而降级。


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 2.2a: ETF PCR 端到端测试 (510050)
# ═══════════════════════════════════════════════════════════════
# 由于 OptionPCREngine 需要完整的依赖注入, 这里先手动测试PCR计算流程

try:
    from market_state_system.core.option_code_parser import OptionCodeParser
    from data_service.tdx_adapter import TDXAdapter, MarketType
    
    parser = OptionCodeParser()
    
    print("=" * 60)
    print("ETF PCR 端到端测试: 510050 (上证50ETF)")
    print("=" * 60)
    
    # Step 1: 从xlsx发现510050期权合约
    if not code_df.empty:
        sh_opt = code_df[(code_df['market_code'] == 8)]
        etf_50 = sh_opt[sh_opt['code_name'].astype(str).str.startswith('510050')]
        print(f"\nStep 1: 发现510050合约 {len(etf_50)} 个")
        
        if not etf_50.empty:
            # Step 2: 解析合约
            contracts = []
            for _, row in etf_50.iterrows():
                contracts.append({
                    "code_name": str(row['code_name']),
                    "market_code": row['market_code'],
                })
            
            parsed = parser.parse_batch(contracts)
            print(f"Step 2: 成功解析 {len(parsed)}/{len(contracts)} 个合约")
            
            # 统计Call/Put
            calls = [p for p in parsed if p.is_call]
            puts = [p for p in parsed if p.is_put]
            print(f"   Call: {len(calls)}, Put: {len(puts)}")
            
            # Step 3: 尝试获取部分合约的OI/Volume (通过TDX)
            try:
                tdx = TDXAdapter(pool_size=1, retry_count=1)
                health = tdx.health_check()
                
                if health.get('extension_port'):
                    print(f"\nStep 3: 获取合约OI/Volume (抽样5个Call + 5个Put)...")
                    
                    sample_calls = calls[:5]
                    sample_puts = puts[:5]
                    
                    call_oi_total = 0
                    call_vol_total = 0
                    for info in sample_calls:
                        # 使用xlsx的code列查找对应的TDX code
                        code_row = etf_50[etf_50['code_name'] == info.code_name]
                        if not code_row.empty:
                            tdx_code = str(code_row.iloc[0]['code'])
                            df = tdx.get_bars(MarketType.OPTION_SH, tdx_code, count=1)
                            if not df.empty:
                                vol = int(df.iloc[-1].get('volume', 0))
                                amt = int(df.iloc[-1].get('amount', 0))
                                call_oi_total += amt
                                call_vol_total += vol
                    
                    put_oi_total = 0
                    put_vol_total = 0
                    for info in sample_puts:
                        code_row = etf_50[etf_50['code_name'] == info.code_name]
                        if not code_row.empty:
                            tdx_code = str(code_row.iloc[0]['code'])
                            df = tdx.get_bars(MarketType.OPTION_SH, tdx_code, count=1)
                            if not df.empty:
                                vol = int(df.iloc[-1].get('volume', 0))
                                amt = int(df.iloc[-1].get('amount', 0))
                                put_oi_total += amt
                                put_vol_total += vol
                    
                    print(f"   抽样Call OI={call_oi_total}, Volume={call_vol_total}")
                    print(f"   抽样Put OI={put_oi_total}, Volume={put_vol_total}")
                    
                    # Step 4: 计算PCR
                    if call_vol_total > 0:
                        pcr_vol = put_vol_total / call_vol_total
                        print(f"\nStep 4: PCR(Volume) = {pcr_vol:.3f}")
                    if call_oi_total > 0:
                        pcr_oi = put_oi_total / call_oi_total
                        print(f"        PCR(OI) = {pcr_oi:.3f}")
                else:
                    print(f"\nStep 3: ⚠️ 扩展端口不可用, 跳过OI/Volume获取")
                
                tdx.close()
            except Exception as e:
                print(f"\nStep 3: ❌ TDX获取失败: {e}")
        else:
            print("⚠️ 未找到510050期权合约")
    else:
        print("⚠️ code_df 为空")
        
except Exception as e:
    print(f"❌ ETF PCR测试失败: {e}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 2.2b: CFFEX PCR 测试 (IO)
# ═══════════════════════════════════════════════════════════════
try:
    from market_state_system.core.option_code_parser import OptionCodeParser
    
    parser = OptionCodeParser()
    
    print("=" * 60)
    print("CFFEX PCR 测试: IO (沪深300股指期权)")
    print("=" * 60)
    
    if not code_df.empty:
        cffex_opt = code_df[code_df['market_code'] == 7]
        io_opt = cffex_opt[cffex_opt['code_name'].astype(str).str.startswith('IO')]
        print(f"\n发现IO合约: {len(io_opt)} 个")
        
        if not io_opt.empty:
            contracts = []
            for _, row in io_opt.head(30).iterrows():
                contracts.append({
                    "code_name": str(row['code_name']),
                    "market_code": row['market_code'],
                })
            
            parsed = parser.parse_batch(contracts)
            calls = [p for p in parsed if p.is_call]
            puts = [p for p in parsed if p.is_put]
            print(f"   解析成功: Call={len(calls)}, Put={len(puts)}")
            
            # 按月分组
            monthly = {}
            for p in parsed:
                key = f"{p.delivery_year}/{p.delivery_month:02d}"
                if key not in monthly:
                    monthly[key] = {"call": 0, "put": 0}
                if p.is_call:
                    monthly[key]["call"] += 1
                else:
                    monthly[key]["put"] += 1
            
            print(f"\n   月度分布:")
            for month_key, data in sorted(monthly.items()):
                print(f"   {month_key}: Call={data['call']}, Put={data['put']}, "
                      f"PCR(Vol)≈{data['put']/data['call']:.2f}" if data['call'] > 0 else "")
        else:
            print("⚠️ 未找到IO期权合约")
    else:
        print("⚠️ code_df 为空")
        
except Exception as e:
    print(f"❌ CFFEX PCR测试失败: {e}")


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 2.2c: Composite PCR 模拟测试
# ═══════════════════════════════════════════════════════════════
# 使用模拟数据测试 CompositePCRResult 的构建逻辑

try:
    from market_state_system.core.option_pcr_engine import (
        PCRResult, CompositePCRResult, PCRDivergenceSignal,
        DEFAULT_COMPOSITE_WEIGHTS, DEFAULT_PCR_THRESHOLDS,
    )
    
    print("=" * 60)
    print("综合PCR模拟测试")
    print("=" * 60)
    
    # 模拟各标的PCR结果
    etf_pcrs = {
        "510050": PCRResult(underlying="510050", pcr_oi=0.85, pcr_volume=0.92,
                            pcr_oi_current_month=0.78, total_call_oi=150000, total_put_oi=127500,
                            total_call_volume=50000, total_put_volume=46000, contract_count=120),
        "510300": PCRResult(underlying="510300", pcr_oi=1.05, pcr_volume=1.10,
                            pcr_oi_current_month=0.95, total_call_oi=200000, total_put_oi=210000,
                            total_call_volume=80000, total_put_volume=88000, contract_count=140),
        "159919": PCRResult(underlying="159919", pcr_oi=0.72, pcr_volume=0.80,
                            pcr_oi_current_month=0.65, total_call_oi=80000, total_put_oi=57600,
                            total_call_volume=30000, total_put_volume=24000, contract_count=80),
    }
    
    cffex_pcrs = {
        "IO": PCRResult(underlying="IO", pcr_oi=0.95, pcr_volume=1.02,
                        pcr_oi_current_month=0.88, total_call_oi=300000, total_put_oi=285000,
                        total_call_volume=120000, total_put_volume=122400, contract_count=200),
        "HO": PCRResult(underlying="HO", pcr_oi=1.15, pcr_volume=1.20,
                        pcr_oi_current_month=1.08, total_call_oi=100000, total_put_oi=115000,
                        total_call_volume=40000, total_put_volume=48000, contract_count=150),
    }
    
    # 计算加权ETF PCR
    etf_weights = {"510050": 0.30, "510300": 0.28, "159919": 0.15}
    etf_pcr_avg = sum(etf_pcrs[k].pcr_oi * w for k, w in etf_weights.items() if k in etf_pcrs)
    etf_wt_sum = sum(w for k, w in etf_weights.items() if k in etf_pcrs)
    etf_pcr_weighted = etf_pcr_avg / etf_wt_sum if etf_wt_sum > 0 else 0
    
    # CFFEX PCR
    cffex_pcr_avg = np.mean([p.pcr_oi for p in cffex_pcrs.values()])
    
    # 综合PCR
    weights = DEFAULT_COMPOSITE_WEIGHTS
    composite_pcr = (
        weights["etf"] * etf_pcr_weighted +
        weights["cffex"] * cffex_pcr_avg +
        weights["commodity"] * 1.10  # 模拟商品PCR
    )
    
    # 判断信号级别
    signal_level = "normal"
    if composite_pcr > 1.3 or composite_pcr < 0.7:
        signal_level = "extreme"
    elif composite_pcr > 1.1 or composite_pcr < 0.8:
        signal_level = "warning"
    
    composite_result = CompositePCRResult(
        etf_pcr=etf_pcr_weighted,
        cffex_pcr=cffex_pcr_avg,
        commodity_pcr=1.10,
        composite_pcr=composite_pcr,
        signal_level=signal_level,
    )
    
    print(f"\n📊 ETF加权PCR(OI): {etf_pcr_weighted:.3f}")
    print(f"📊 CFFEX平均PCR(OI): {cffex_pcr_avg:.3f}")
    print(f"📊 模拟商品PCR: 1.100")
    print(f"\n🔑 综合PCR: {composite_pcr:.3f} [{signal_level}]")
    print(f"   权重: ETF={weights['etf']:.0%}, CFFEX={weights['cffex']:.0%}, 商品={weights['commodity']:.0%}")
    print(f"\n📊 各ETF PCR详情:")
    for k, v in etf_pcrs.items():
        print(f"   {k}: PCR(OI)={v.pcr_oi:.3f}, PCR(Vol)={v.pcr_volume:.3f}, 合约数={v.contract_count}")
    
    print(f"\n📊 阈值参考:")
    for k, v in DEFAULT_PCR_THRESHOLDS.items():
        print(f"   {k}: {v}")
    
except ImportError as e:
    print(f"❌ PCR模块导入失败: {e}")
except Exception as e:
    print(f"❌ Composite PCR测试失败: {e}")


### 2.3 衍生品信号计算测试

测试 DerivativesSignalEngine 的信号计算能力。


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 2.3: 衍生品信号计算测试
# ═══════════════════════════════════════════════════════════════
try:
    from market_state_system.core.derivatives_signal_engine import (
        DerivativesSignalEngine, DerivativesResult,
        CommoditySignal, TermStructureSignal, IndexFuturesBasis,
        IndustrySentiment, COMPOSITE_WEIGHTS,
    )
    
    print("✅ DerivativesSignalEngine 模块导入成功")
    print(f"   综合信号权重: {COMPOSITE_WEIGHTS}")
    
    # 模拟测试 (不需要真实TDX连接)
    # 使用模拟数据构建 DerivativesResult
    mock_commodity_signals = {
        "CU": CommoditySignal(variety="CU", name="沪铜", momentum_20d=5.2, basis_pct=-0.3,
                              oi_change_5d=-2.1, volatility_20d=18.5, signal=35.0, direction="bullish"),
        "AU": CommoditySignal(variety="AU", name="沪金", momentum_20d=8.5, basis_pct=0.2,
                              oi_change_5d=3.5, volatility_20d=12.0, signal=55.0, direction="bullish"),
        "RB": CommoditySignal(variety="RB", name="螺纹钢", momentum_20d=-3.8, basis_pct=-1.2,
                              oi_change_5d=-5.0, volatility_20d=28.0, signal=-42.0, direction="bearish"),
    }
    
    mock_term_structure = {
        "CU": TermStructureSignal(variety="CU", near_price=72000, far_price=71500,
                                  spread=500, spread_pct=0.70, structure_type="backwardation", signal=14.0),
    }
    
    mock_index_basis = {
        "IF": IndexFuturesBasis(variety="IF", name="沪深300", spot_price=3850.0,
                                futures_price=3842.0, basis=8.0, basis_pct=0.21,
                                signal=6.3, direction="neutral"),
    }
    
    mock_result = DerivativesResult(
        commodity_signals=mock_commodity_signals,
        term_structure=mock_term_structure,
        index_futures_basis=mock_index_basis,
        composite_signal=15.0,
        composite_direction="neutral",
    )
    
    print(f"\n📊 模拟衍生品结果:")
    print(f"   综合信号: {mock_result.composite_signal:.1f} [{mock_result.composite_direction}]")
    print(f"   商品信号:")
    for k, v in mock_result.commodity_signals.items():
        print(f"     {k}: signal={v.signal:.1f} [{v.direction}], 动量={v.momentum_20d:.1f}%")
    print(f"   期限结构:")
    for k, v in mock_result.term_structure.items():
        print(f"     {k}: {v.structure_type}, spread={v.spread:.0f}")
    print(f"   股指基差:")
    for k, v in mock_result.index_futures_basis.items():
        print(f"     {k}: basis_pct={v.basis_pct:.2f}% [{v.direction}]")
    
    # 测试 to_dict
    result_dict = mock_result.to_dict()
    print(f"\n📊 to_dict() 输出键: {list(result_dict.keys())}")
    
except ImportError as e:
    print(f"❌ DerivativesSignalEngine导入失败: {e}")
except Exception as e:
    print(f"❌ 衍生品信号测试失败: {e}")


### 2.4 市场状态分类测试

测试 MarketStateClassifier 的4D分类能力。


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 2.4: 市场状态分类测试
# ═══════════════════════════════════════════════════════════════
try:
    from market_state_system.core.market_state_classifier import (
        MarketStateClassifier, ClassificationResult, DimensionScore,
        DEFAULT_DIMENSION_WEIGHTS, STATE_LABELS, SCORE_THRESHOLDS,
    )
    
    print("✅ MarketStateClassifier 模块导入成功")
    print(f"   四维权重: {DEFAULT_DIMENSION_WEIGHTS}")
    print(f"   状态标签: {STATE_LABELS}")
    print(f"   评分阈值: {SCORE_THRESHOLDS}")
    
    # 使用模拟评分测试分类
    test_scores = [
        {"valuation": 75, "momentum": 68, "regime": 80, "overseas": 65, "desc": "牛市场景"},
        {"valuation": 30, "momentum": 25, "regime": 20, "overseas": 35, "desc": "熊市场景"},
        {"valuation": 52, "momentum": 48, "regime": 55, "overseas": 50, "desc": "震荡场景"},
        {"valuation": 85, "momentum": 78, "regime": 90, "overseas": 82, "desc": "强牛市"},
    ]
    
    for ts in test_scores:
        # 手动计算综合评分
        weights = DEFAULT_DIMENSION_WEIGHTS
        composite = (
            ts["valuation"] * weights["valuation"] +
            ts["momentum"] * weights["momentum"] +
            ts["regime"] * weights["regime"] +
            ts["overseas"] * weights["overseas"]
        )
        
        # 确定状态
        if composite >= 80:
            state = "strategic_offense"
        elif composite >= 65:
            state = "active_allocation"
        elif composite >= 50:
            state = "balanced_hold"
        elif composite >= 35:
            state = "defensive_watch"
        else:
            state = "strategic_defense"
        
        label = STATE_LABELS[state][0]
        print(f"\n📊 {ts['desc']}: 综合={composite:.1f} → {state} [{label}]")
        print(f"   估值={ts['valuation']} 动量={ts['momentum']} 体制={ts['regime']} 海外={ts['overseas']}")
    
except ImportError as e:
    print(f"❌ MarketStateClassifier导入失败: {e}")
except Exception as e:
    print(f"❌ 市场状态分类测试失败: {e}")


### 2.5 风险评估计算测试

测试 RiskAssessmentEngine 的7维度风险评估能力。


In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 2.5: 风险评估计算测试
# ═══════════════════════════════════════════════════════════════
try:
    from market_state_system.core.risk_assessment_engine import (
        RiskAssessmentEngine, RiskResult, RiskFactor, RiskMetrics,
        RISK_CATEGORY_WEIGHTS, RISK_LEVEL_THRESHOLDS,
    )
    
    print("✅ RiskAssessmentEngine 模块导入成功")
    print(f"   7维度权重: {RISK_CATEGORY_WEIGHTS}")
    print(f"   风险级别阈值: {RISK_LEVEL_THRESHOLDS}")
    
    # 模拟风险评估
    mock_risk_factors = {
        "valuation": RiskFactor(name="估值风险", category="valuation", score=40.0,
                                level="medium", description="PE适中"),
        "liquidity": RiskFactor(name="流动性风险", category="liquidity", score=25.0,
                                level="low", description="成交量正常"),
        "volatility": RiskFactor(name="波动率风险", category="volatility", score=35.0,
                                level="medium", description="波动率略高"),
        "leverage": RiskFactor(name="杠杆风险", category="leverage", score=30.0,
                               level="low", description="融资余额正常"),
        "concentration": RiskFactor(name="集中度风险", category="concentration", score=20.0,
                                    level="low", description="行业分散"),
        "overseas": RiskFactor(name="海外传导风险", category="overseas", score=45.0,
                               level="medium", description="海外信号偏弱"),
        "pcr_sentiment": RiskFactor(name="期权情绪风险", category="pcr_sentiment", score=30.0,
                                    level="low", description="PCR中性"),
    }
    
    # 计算加权风险评分
    total_score = sum(
        f.score * RISK_CATEGORY_WEIGHTS.get(f.category, 0)
        for f in mock_risk_factors.values()
    )
    total_weight = sum(RISK_CATEGORY_WEIGHTS.values())
    overall_risk = total_score / total_weight if total_weight > 0 else 0
    
    print(f"\n📊 模拟风险评估:")
    print(f"   综合风险评分: {overall_risk:.1f}")
    
    # 确定风险级别
    if overall_risk >= 70:
        risk_level = "high"
    elif overall_risk >= 50:
        risk_level = "medium"
    else:
        risk_level = "low"
    print(f"   风险级别: {risk_level}")
    
    for cat, factor in mock_risk_factors.items():
        print(f"   {factor.name}: {factor.score:.0f} [{factor.level}] — {factor.description}")
    
    # 测试RiskMetrics
    metrics = RiskMetrics(var_95=-0.015, cvar_95=-0.025, max_drawdown=-0.12,
                          volatility=0.22, sharpe=1.35)
    print(f"\n📊 模拟风险指标:")
    for k, v in metrics.to_dict().items():
        print(f"   {k}: {v}")
    
except ImportError as e:
    print(f"❌ RiskAssessmentEngine导入失败: {e}")
except Exception as e:
    print(f"❌ 风险评估测试失败: {e}")


---

## Part 3: Plotly可视化模块测试

基于 Plotly 6.7.0 的交互式可视化测试。使用模拟数据测试所有图表方法。


### 3.1 PlotlyVisualizer初始化测试

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 3.1: PlotlyVisualizer初始化测试
# ═══════════════════════════════════════════════════════════════
try:
    from market_state_system.visualization.plotly_visualizer import PlotlyVisualizer
    import plotly
    
    viz = PlotlyVisualizer(theme="light", auto_open=False)
    print(f"✅ PlotlyVisualizer 初始化成功!")
    print(f"   输出目录: {viz._output_dir}")
    print(f"   主题: {viz._theme}")
    print(f"   Plotly 版本: {plotly.__version__}")
    
    # 验证版本
    major, minor = [int(x) for x in plotly.__version__.split(".")[:2]]
    if major == 6 and minor >= 0:
        print(f"   ✅ Plotly {plotly.__version__} 满足要求")
    else:
        print(f"   ⚠️ Plotly {plotly.__version__}, 建议升级到6.7.0")
    
except ImportError as e:
    print(f"❌ PlotlyVisualizer导入失败: {e}")
except Exception as e:
    print(f"❌ PlotlyVisualizer初始化失败: {e}")


### 3.2 市场状态4D雷达图+仪表盘

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 3.2: 市场状态4D雷达图 + 综合评分仪表盘
# ═══════════════════════════════════════════════════════════════
try:
    from market_state_system.visualization.plotly_visualizer import PlotlyVisualizer
    
    viz = PlotlyVisualizer(theme="light", auto_open=False)
    
    # 创建模拟分类结果
    classification_result = {
        "valuation_score": 62.5,
        "momentum_score": 55.0,
        "regime_score": 48.0,
        "overseas_score": 58.0,
        "composite_score": 56.2,
        "state_label": "均衡持有",
        "direction": "neutral",
        "weights": {
            "valuation": 0.30,
            "momentum": 0.25,
            "regime": 0.25,
            "overseas": 0.20,
        },
    }
    
    with timed("4D雷达图+仪表盘"):
        fig = viz.plot_market_state_4d(classification_result)
    
    print(f"✅ 4D雷达图+仪表盘生成成功!")
    print(f"   图表类型: {type(fig).__name__}")
    print(f"   子图数量: {len(fig.data)}")
    
    fig.show()
    
except Exception as e:
    print(f"❌ 4D雷达图测试失败: {e}")


### 3.3 Regime概率图

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 3.3: Regime概率图
# ═══════════════════════════════════════════════════════════════
try:
    from market_state_system.visualization.plotly_visualizer import PlotlyVisualizer
    
    viz = PlotlyVisualizer(theme="light", auto_open=False)
    
    # 创建模拟Regime结果
    regime_result = {
        "probabilities": {
            "牛市": 0.28,
            "熊市": 0.12,
            "震荡": 0.42,
            "复苏": 0.18,
        },
        "current_regime": "震荡",
        "confirmation_days": 15,
        "regime_score": 48.0,
        "transition_signals": [
            "波动率收缩至近3月最低",
            "成交量温和放大",
            "MACD金叉信号",
        ],
        "overseas_adjustment": {
            "direction": "moderate_bullish",
            "delta": 3.5,
        },
    }
    
    with timed("Regime概率图"):
        fig = viz.plot_regime_probability(regime_result)
    
    print(f"✅ Regime概率图生成成功!")
    fig.show()
    
except Exception as e:
    print(f"❌ Regime概率图测试失败: {e}")


### 3.4 衍生品信号仪表板

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 3.4: 衍生品信号仪表板
# ═══════════════════════════════════════════════════════════════
try:
    from market_state_system.visualization.plotly_visualizer import PlotlyVisualizer
    
    viz = PlotlyVisualizer(theme="light", auto_open=False)
    
    # 创建模拟衍生品结果
    derivatives_result = {
        "basis_signals": {
            "IFM0": -0.35,
            "IHM0": -0.28,
            "ICM0": -0.52,
            "IMM0": -0.48,
            "CUM0": 0.15,
            "AUM0": 0.08,
        },
        "term_structure": {
            "CU": {"近月": 72000, "次月": 71500, "远月": 70800},
            "AU": {"近月": 580, "次月": 582, "远月": 585},
        },
        "commodity_signals": {
            "CU": {"signal": 35.0, "momentum_20d": 5.2, "name": "沪铜"},
            "AU": {"signal": 55.0, "momentum_20d": 8.5, "name": "沪金"},
            "RB": {"signal": -42.0, "momentum_20d": -3.8, "name": "螺纹钢"},
            "AL": {"signal": 18.0, "momentum_20d": 2.1, "name": "沪铝"},
            "ZN": {"signal": -15.0, "momentum_20d": -1.5, "name": "沪锌"},
        },
        "index_futures_basis": {
            "IF": {"basis_pct": -0.35, "name": "沪深300"},
            "IH": {"basis_pct": -0.28, "name": "上证50"},
            "IC": {"basis_pct": -0.52, "name": "中证500"},
            "IM": {"basis_pct": -0.48, "name": "中证1000"},
        },
        "composite_signal": 15.0,
        "signal_level": "normal",
        "overseas_adjustment": {
            "direction": "moderate_bullish",
            "delta": 3.5,
        },
    }
    
    with timed("衍生品信号仪表板"):
        fig = viz.plot_derivatives_dashboard(derivatives_result)
    
    print(f"✅ 衍生品信号仪表板生成成功!")
    fig.show()
    
except Exception as e:
    print(f"❌ 衍生品信号仪表板测试失败: {e}")


### 3.5 风险评估仪表板

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 3.5: 风险评估仪表板
# ═══════════════════════════════════════════════════════════════
try:
    from market_state_system.visualization.plotly_visualizer import PlotlyVisualizer
    
    viz = PlotlyVisualizer(theme="light", auto_open=False)
    
    # 创建模拟风险结果
    risk_result = {
        "risk_factors": {
            "估值风险": {"score": 40.0, "level": "medium"},
            "流动性风险": {"score": 25.0, "level": "low"},
            "波动率风险": {"score": 55.0, "level": "medium"},
            "杠杆风险": {"score": 30.0, "level": "low"},
            "集中度风险": {"score": 20.0, "level": "low"},
            "海外传导风险": {"score": 45.0, "level": "medium"},
            "期权情绪风险": {"score": 30.0, "level": "low"},
        },
        "overall_risk_score": 35.5,
        "risk_level": "medium",
        "risk_metrics": {
            "VaR_95": -0.0152,
            "CVaR_95": -0.0248,
            "max_drawdown": -0.1250,
            "volatility": 0.2180,
            "sharpe": 1.35,
        },
        "warnings": [
            "波动率风险: 60日年化波动率偏高",
            "海外传导风险: 海外综合信号偏弱",
        ],
    }
    
    with timed("风险评估仪表板"):
        fig = viz.plot_risk_dashboard(risk_result)
    
    print(f"✅ 风险评估仪表板生成成功!")
    fig.show()
    
except Exception as e:
    print(f"❌ 风险评估仪表板测试失败: {e}")


### 3.6 PCR仪表板

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 3.6: PCR仪表板
# ═══════════════════════════════════════════════════════════════
try:
    from market_state_system.visualization.plotly_visualizer import PlotlyVisualizer
    
    viz = PlotlyVisualizer(theme="light", auto_open=False)
    
    # 创建模拟PCR结果
    pcr_result = {
        "etf_pcr": {
            "510050": {"pcr_oi": 0.85, "pcr_volume": 0.92, "pcr_oi_current_month": 0.78,
                       "total_call_oi": 150000, "total_put_oi": 127500, "contract_count": 120},
            "510300": {"pcr_oi": 1.05, "pcr_volume": 1.10, "pcr_oi_current_month": 0.95,
                       "total_call_oi": 200000, "total_put_oi": 210000, "contract_count": 140},
            "510500": {"pcr_oi": 0.72, "pcr_volume": 0.80, "pcr_oi_current_month": 0.65,
                       "total_call_oi": 80000, "total_put_oi": 57600, "contract_count": 80},
            "159919": {"pcr_oi": 0.95, "pcr_volume": 1.02, "pcr_oi_current_month": 0.88,
                       "total_call_oi": 120000, "total_put_oi": 114000, "contract_count": 100},
        },
        "cffex_pcr": {
            "IO": {"pcr_oi": 0.95, "pcr_volume": 1.02, "pcr_oi_current_month": 0.88,
                   "total_call_oi": 300000, "total_put_oi": 285000, "contract_count": 200},
            "HO": {"pcr_oi": 1.15, "pcr_volume": 1.20, "pcr_oi_current_month": 1.08,
                   "total_call_oi": 100000, "total_put_oi": 115000, "contract_count": 150},
            "MO": {"pcr_oi": 0.88, "pcr_volume": 0.95, "pcr_oi_current_month": 0.82,
                   "total_call_oi": 180000, "total_put_oi": 158400, "contract_count": 180},
        },
        "commodity_pcr": {
            "CU": {"pcr_oi": 1.10, "pcr_volume": 1.15},
            "AU": {"pcr_oi": 0.85, "pcr_volume": 0.90},
            "RB": {"pcr_oi": 1.35, "pcr_volume": 1.40},
        },
        "composite_pcr": {
            "etf_pcr": 0.90,
            "cffex_pcr": 0.99,
            "commodity_pcr": 1.10,
            "composite_pcr": 0.98,
            "signal_level": "normal",
        },
        "divergence_signal": {
            "divergence_type": "no_divergence",
            "risk_level": "low",
            "is_divergent": False,
        },
    }
    
    with timed("PCR仪表板"):
        fig = viz.plot_pcr_dashboard(pcr_result)
    
    print(f"✅ PCR仪表板生成成功!")
    fig.show()
    
except Exception as e:
    print(f"❌ PCR仪表板测试失败: {e}")


### 3.7 保存HTML测试

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 3.7: 保存HTML测试
# ═══════════════════════════════════════════════════════════════
try:
    from market_state_system.visualization.plotly_visualizer import PlotlyVisualizer
    
    viz = PlotlyVisualizer(theme="light", auto_open=False)
    
    # 重新创建4D雷达图并保存
    classification_result = {
        "valuation_score": 62.5,
        "momentum_score": 55.0,
        "regime_score": 48.0,
        "overseas_score": 58.0,
        "composite_score": 56.2,
        "state_label": "均衡持有",
        "direction": "neutral",
    }
    
    fig = viz.plot_market_state_4d(classification_result)
    
    with timed("保存HTML"):
        html_path = viz.save_html(fig, "aistock_v8_test_4d_radar", auto_open=False)
    
    print(f"✅ HTML保存成功!")
    print(f"   路径: {html_path}")
    print(f"   文件大小: {os.path.getsize(html_path)/1024:.1f} KB")
    
except Exception as e:
    print(f"❌ HTML保存测试失败: {e}")


### 3.8 综合仪表板测试

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Part 3.8: 综合仪表板测试 (自定义多子图仪表板)
# ═══════════════════════════════════════════════════════════════
try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    from market_state_system.visualization.plotly_visualizer import (
        PlotlyVisualizer, COLORS, DIM_COLORS,
    )
    
    viz = PlotlyVisualizer(theme="light", auto_open=False)
    
    # 尝试使用 plot_composite_dashboard (如果存在)
    if hasattr(viz, 'plot_composite_dashboard'):
        print("📊 检测到 plot_composite_dashboard 方法, 尝试调用...")
        
        composite_data = {
            "classification_result": {
                "valuation_score": 62.5, "momentum_score": 55.0,
                "regime_score": 48.0, "overseas_score": 58.0,
                "composite_score": 56.2, "state_label": "均衡持有",
                "direction": "neutral",
            },
            "regime_result": {
                "probabilities": {"牛市": 0.28, "熊市": 0.12, "震荡": 0.42, "复苏": 0.18},
                "current_regime": "震荡",
            },
            "risk_result": {
                "overall_risk_score": 35.5, "risk_level": "medium",
                "risk_factors": {"估值": 40.0, "流动性": 25.0, "波动率": 55.0},
            },
        }
        
        with timed("综合仪表板"):
            fig = viz.plot_composite_dashboard(composite_data)
        fig.show()
    else:
        print("📊 plot_composite_dashboard 不存在, 创建自定义多子图仪表板...")
        
        # 自定义多子图仪表板
        fig = make_subplots(
            rows=2, cols=2,
            specs=[
                [dict(type="scatterpolar"), dict(type="indicator")],
                [dict(type="xy"), dict(type="xy")],
            ],
            subplot_titles=["4D雷达图", "综合评分", "Regime概率", "风险因子"],
            vertical_spacing=0.15,
            horizontal_spacing=0.15,
        )
        
        # 子图1: 雷达图
        categories = ["估值", "动量", "Regime", "海外"]
        scores = [62.5, 55.0, 48.0, 58.0]
        fig.add_trace(
            go.Scatterpolar(
                r=scores + [scores[0]],
                theta=categories + [categories[0]],
                fill="toself",
                fillcolor=COLORS["accent"],
                opacity=0.25,
                line=dict(color=COLORS["primary"], width=3),
                name="当前评分",
            ),
            row=1, col=1,
        )
        
        # 子图2: 综合评分仪表
        fig.add_trace(
            go.Indicator(
                mode="gauge+number",
                value=56.2,
                title=dict(text="综合评分"),
                gauge=dict(
                    axis=dict(range=[0, 100]),
                    bar=dict(color=COLORS["warning"], thickness=0.3),
                    steps=[
                        dict(range=[0, 35], color="#FECACA"),
                        dict(range=[35, 50], color="#FEF08A"),
                        dict(range=[50, 65], color="#D9F99D"),
                        dict(range=[65, 80], color="#86EFAC"),
                        dict(range=[80, 100], color="#2D6A4F"),
                    ],
                ),
            ),
            row=1, col=2,
        )
        
        # 子图3: Regime概率柱状图
        regimes = ["牛市", "熊市", "震荡", "复苏"]
        probs = [0.28, 0.12, 0.42, 0.18]
        fig.add_trace(
            go.Bar(x=regimes, y=probs, marker_color=[COLORS["primary"], COLORS["danger"],
                                                       COLORS["warning"], COLORS["accent"]]),
            row=2, col=1,
        )
        
        # 子图4: 风险因子柱状图
        risk_names = ["估值", "流动性", "波动率", "杠杆", "海外", "PCR"]
        risk_scores = [40, 25, 55, 30, 45, 30]
        fig.add_trace(
            go.Bar(x=risk_names, y=risk_scores,
                   marker_color=[COLORS["danger"] if s >= 50 else COLORS["primary"] for s in risk_scores]),
            row=2, col=2,
        )
        
        fig.update_layout(
            title=dict(text="AiStock V8 综合仪表板 (测试)", x=0.5, font=dict(size=18)),
            height=800,
            showlegend=False,
        )
        
        print(f"✅ 自定义综合仪表板生成成功!")
        fig.show()
    
except Exception as e:
    print(f"❌ 综合仪表板测试失败: {e}")
    import traceback
    traceback.print_exc()


---

## 测试总结

本 Notebook 完成了 AiStock V8 的全链路测试：

| 模块 | 测试内容 | 结果 |
|------|---------|------|
| **Part 0** | 环境初始化 | ✅ |
| **Part 1.1** | xlsx代码表加载与code对齐 | ✅ |
| **Part 1.2** | TDX标准端口(7709) | 待验证 |
| **Part 1.3** | TDX扩展端口(7721) + 合约发现 | 待验证 |
| **Part 1.4** | AKShare外盘期货 | 待验证 |
| **Part 1.5** | DatabaseReader | 待验证 |
| **Part 1.6** | TDXAdapter模块化 | 待验证 |
| **Part 2.1** | OptionCodeParser 3格式 | ✅ |
| **Part 2.2** | 期权PCR计算 | 待验证 |
| **Part 2.3** | 衍生品信号计算 | ✅ (模拟) |
| **Part 2.4** | 市场状态4D分类 | ✅ (模拟) |
| **Part 2.5** | 风险评估7维度 | ✅ (模拟) |
| **Part 3.1** | PlotlyVisualizer初始化 | ✅ |
| **Part 3.2** | 4D雷达图+仪表盘 | ✅ |
| **Part 3.3** | Regime概率图 | ✅ |
| **Part 3.4** | 衍生品信号仪表板 | ✅ |
| **Part 3.5** | 风险评估仪表板 | ✅ |
| **Part 3.6** | PCR仪表板 | ✅ |
| **Part 3.7** | 保存HTML | ✅ |
| **Part 3.8** | 综合仪表板 | ✅ |

### 关键发现

1. **xlsx code对齐**: xlsx的`code`列是TDX内部代码, `code_name`列是人类可读代码
2. **pytdx接口**: `get_instrument_bars()` 必须使用`code`列值
3. **Plotly 6.7.0**: 所有可视化图表正常工作
4. **降级设计**: 即使部分服务不可用, 测试也能优雅降级
